In [ ]:
# ================================================================
# LIMPEZA, AUDITORIA E PREPARAÇÃO DA BASE ANALÍTICA
# ================================================================

from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd


# ----------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ----------------------------------------------------------------

from pathlib import Path

OUTPUT_DIR = Path("eda/outputs/step_03_data_cleaning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_PATH = OUTPUT_DIR / "missing_status_audit.parquet"
AUDIT_SUMMARY_PATH = OUTPUT_DIR / "missing_status_summary.parquet"
ANALYTICAL_PATH = OUTPUT_DIR / "analytical_dataset.parquet"
CLUSTERING_PATH = OUTPUT_DIR / "clustering_dataset.parquet"


# ----------------------------------------------------------------
# 2. REPRESENTAÇÕES DE AUSÊNCIA E VALORES ESPECIAIS
# ----------------------------------------------------------------

# None, np.nan, pd.NA e NaT são identificados diretamente por pd.isna().
# Após a normalização, nan, NaN e NAN passam a ser "NAN".

REAL_MISSING = {
    "",
    "NULO",
    "NULL",
    "N/D",
    "NAN",
}

INFORMATIONAL_MISSING = {
    "NÃO INFORMADO",
}

OPERATIONAL_REPLACEMENTS = {
    "ATENDIMENTO INTERRPOMPIDO": "ATENDIMENTO INTERROMPIDO",
}

OPERATIONAL_MISSING = {
    "ATENDIMENTO INTERROMPIDO",
}

UNKNOWN_VALUE = {
    "NÃO SABE",
    "NÃO SABE INFORMAR",
    "DENUNCIANTE NÃO SOUBE INFORMAR",
}

# Há informação substantiva parcial, portanto não vira pd.NA integralmente.
PARTIALLY_KNOWN = {
    "TEM DEFICIÊNCIA MAS NÃO SABE QUAL":
        "DEFICIÊNCIA NÃO ESPECIFICADA",
}

# Não aplicável não é dado faltante.
NOT_APPLICABLE = {
    "NÃO SE APLICA - VÍTIMA COMUNIDADE/FAMÍLIA":
        "NÃO SE APLICA",
}


# ----------------------------------------------------------------
# 3. NORMALIZAÇÃO DE TEXTO
# ----------------------------------------------------------------

def normalize_text(value):
    """
    Normaliza strings sem alterar valores ausentes nativos.

    Procedimentos:
    - converte Unicode para NFKC;
    - remove espaços nas extremidades;
    - converte para maiúsculas;
    - remove ? e *;
    - padroniza espaços, barras e parênteses.
    """

    if pd.isna(value):
        return pd.NA

    value = unicodedata.normalize("NFKC", str(value))
    value = value.upper().strip()

    value = value.replace("?", "").replace("*", "")
    value = re.sub(r"\s+", " ", value)
    value = re.sub(r"\s+\(A)", " (A)", value)
    value = re.sub(r"\s*/\s*", " / ", value)

    return value


# ----------------------------------------------------------------
# 4. CLASSIFICAÇÃO DE UMA OBSERVAÇÃO
# ----------------------------------------------------------------

def classify_observation(value):
    """
    Retorna o valor analítico, a classe e o status da observação.
    """

    if pd.isna(value):
        return {
            "valor_normalizado": pd.NA,
            "valor_analitico": pd.NA,
            "classe": "AUSENTE_REAL",
            "status": "AUSÊNCIA_TÉCNICA_NATIVA",
            "observado": False,
            "aplicavel": True,
        }

    normalized = normalize_text(value)

    # Corrige variantes inequívocas antes da classificação.
    normalized = OPERATIONAL_REPLACEMENTS.get(
        normalized,
        normalized,
    )

    if normalized in REAL_MISSING:
        return {
            "valor_normalizado": normalized,
            "valor_analitico": pd.NA,
            "classe": "AUSENTE_REAL",
            "status": "AUSÊNCIA_TÉCNICA",
            "observado": False,
            "aplicavel": True,
        }

    if normalized in INFORMATIONAL_MISSING:
        return {
            "valor_normalizado": normalized,
            "valor_analitico": pd.NA,
            "classe": "AUSENTE_INFORMACIONAL",
            "status": "NÃO_INFORMADO",
            "observado": False,
            "aplicavel": True,
        }

    if normalized in OPERATIONAL_MISSING:
        return {
            "valor_normalizado": normalized,
            "valor_analitico": pd.NA,
            "classe": "AUSENTE_OPERACIONAL",
            "status": "ATENDIMENTO_INTERROMPIDO",
            "observado": False,
            "aplicavel": True,
        }

    if normalized in UNKNOWN_VALUE:
        return {
            "valor_normalizado": normalized,
            "valor_analitico": pd.NA,
            "classe": "DESCONHECIDO",
            "status": "DESCONHECIDO_PELO_INFORMANTE",
            "observado": False,
            "aplicavel": True,
        }

    if normalized in PARTIALLY_KNOWN:
        return {
            "valor_normalizado": normalized,
            "valor_analitico": PARTIALLY_KNOWN[normalized],
            "classe": "INFORMAÇÃO_PARCIAL",
            "status": "CARACTERÍSTICA_NÃO_ESPECIFICADA",
            "observado": True,
            "aplicavel": True,
        }

    if normalized in NOT_APPLICABLE:
        return {
            "valor_normalizado": normalized,
            "valor_analitico": NOT_APPLICABLE[normalized],
            "classe": "NÃO_APLICÁVEL",
            "status": "NÃO_APLICÁVEL",
            "observado": False,
            "aplicavel": False,
        }

    return {
        "valor_normalizado": normalized,
        "valor_analitico": normalized,
        "classe": "OBSERVADO",
        "status": "VALOR_SUBSTANTIVO",
        "observado": True,
        "aplicavel": True,
    }


# ----------------------------------------------------------------
# 5. SELEÇÃO DAS COLUNAS TEXTUAIS
# ----------------------------------------------------------------

def get_text_columns(df):
    """Seleciona colunas textuais e categóricas."""

    return df.select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()


# ----------------------------------------------------------------
# 6. TABELA DE AUDITORIA
# ----------------------------------------------------------------

def build_audit_table(df, columns=None):
    """
    Gera uma linha para cada combinação de:
    variável, valor normalizado, classe e status.
    """

    if columns is None:
        columns = get_text_columns(df)

    audit_parts = []

    for column in columns:
        classified = pd.DataFrame(
            df[column]
            .apply(classify_observation)
            .tolist(),
            index=df.index,
        )

        classified.insert(0, "variavel", column)

        grouped = (
            classified
            .groupby(
                [
                    "variavel",
                    "valor_normalizado",
                    "classe",
                    "status",
                    "observado",
                    "aplicavel",
                ],
                dropna=False,
                observed=True,
            )
            .size()
            .rename("frequencia")
            .reset_index()
        )

        grouped["total_variavel"] = len(df)

        grouped["percentual_total"] = (
            grouped["frequencia"]
            .div(grouped["total_variavel"])
            .mul(100)
            .round(4)
        )

        audit_parts.append(grouped)

    if not audit_parts:
        return pd.DataFrame()

    audit = pd.concat(audit_parts, ignore_index=True)

    return audit.sort_values(
        ["variavel", "frequencia"],
        ascending=[True, False],
        ignore_index=True,
    )


# ----------------------------------------------------------------
# 7. LIMPEZA DA BASE ANALÍTICA
# ----------------------------------------------------------------

def clean_analytical_dataset(
    df,
    columns=None,
    add_missing_indicators=True,
):
    """
    Cria uma cópia analítica da base.

    - normaliza colunas textuais;
    - converte ausências em pd.NA;
    - preserva valores não aplicáveis;
    - preserva informação parcial;
    - opcionalmente cria indicadores binários de ausência.
    """

    cleaned = df.copy(deep=True)

    if columns is None:
        columns = get_text_columns(cleaned)

    for column in columns:
        classified = pd.DataFrame(
            cleaned[column]
            .apply(classify_observation)
            .tolist(),
            index=cleaned.index,
        )

        cleaned[column] = classified["valor_analitico"]

        if add_missing_indicators:
            cleaned[f"{column}__missing"] = (
                ~classified["observado"]
                & classified["aplicavel"]
            ).astype("int8")

            cleaned[f"{column}__not_applicable"] = (
                ~classified["aplicavel"]
            ).astype("int8")

    return cleaned


# ----------------------------------------------------------------
# 8. RESUMO DA QUALIDADE POR VARIÁVEL
# ----------------------------------------------------------------

def summarize_audit(audit):
    """
    Consolida a auditoria por variável e classe.
    """

    if audit.empty:
        return pd.DataFrame()

    summary = (
        audit
        .pivot_table(
            index="variavel",
            columns="classe",
            values="frequencia",
            aggfunc="sum",
            fill_value=0,
            observed=True,
        )
        .reset_index()
    )

    status_counts = (
        audit.groupby("variavel", observed=True)["frequencia"]
        .sum()
        .rename("total_registros")
    )

    summary = summary.merge(
        status_counts,
        on="variavel",
        how="left",
    )

    not_applicable = summary.get(
        "NÃO_APLICÁVEL",
        pd.Series(0, index=summary.index),
    )

    observed = summary.get(
        "OBSERVADO",
        pd.Series(0, index=summary.index),
    )

    partially_known = summary.get(
        "INFORMAÇÃO_PARCIAL",
        pd.Series(0, index=summary.index),
    )

    summary["total_aplicavel"] = (
        summary["total_registros"] - not_applicable
    )

    summary["total_disponivel"] = (
        observed + partially_known
    )

    summary["completude_aplicavel_pct"] = np.where(
        summary["total_aplicavel"] > 0,
        (
            100
            * summary["total_disponivel"]
            / summary["total_aplicavel"]
        ),
        np.nan,
    )

    summary["completude_aplicavel_pct"] = (
        summary["completude_aplicavel_pct"].round(2)
    )

    return summary.sort_values(
        "completude_aplicavel_pct",
        ascending=True,
        na_position="last",
        ignore_index=True,
    )


# ----------------------------------------------------------------
# 9. BASE AUXILIAR PARA AGRUPAMENTOS
# ----------------------------------------------------------------

def prepare_clustering_dataset(df_analytical, id_columns=None):
    """
    Prepara uma base inicial para análise de agrupamentos.

    Não executa one-hot encoding nem imputação. Apenas:
    - remove identificadores;
    - remove colunas completamente vazias;
    - remove colunas constantes;
    - preserva indicadores de ausência.
    """

    clustering = df_analytical.copy()

    if id_columns:
        clustering = clustering.drop(
            columns=id_columns,
            errors="ignore",
        )

    # Remove colunas totalmente ausentes.
    clustering = clustering.dropna(
        axis=1,
        how="all",
    )

    # Remove colunas sem variabilidade.
    nunique = clustering.nunique(
        dropna=False,
    )

    constant_columns = nunique[nunique <= 1].index

    clustering = clustering.drop(
        columns=constant_columns,
        errors="ignore",
    )

    return clustering


# ----------------------------------------------------------------
# 10. EXECUÇÃO
# ----------------------------------------------------------------

# Pressupõe que o DataFrame original se chama df.
# A variável hash deve ser preservada na base analítica, mas retirada
# da matriz usada para calcular distâncias.

text_columns = get_text_columns(df_raw)

audit_detail = build_audit_table(
    df=df_raw,
    columns=text_columns,
)

audit_summary = summarize_audit(
    audit=audit_detail,
)

df_analytical = clean_analytical_dataset(
    df=df_raw,
    columns=text_columns,
    add_missing_indicators=True,
)

df_clustering = prepare_clustering_dataset(
    df_analytical=df_analytical,
    id_columns=[
        "hash",
        # Acrescente outros identificadores, se existirem:
        # "id_denuncia",
        # "protocolo",
    ],
)


PARQUET_OPTIONS = {
    "engine": "pyarrow",
    "compression": "zstd",
    "index": False,
}

audit_detail.to_parquet(
    AUDIT_PATH,
    **PARQUET_OPTIONS,
)

audit_summary.to_parquet(
    AUDIT_SUMMARY_PATH,
    **PARQUET_OPTIONS,
)

df_analytical.to_parquet(
    ANALYTICAL_PATH,
    **PARQUET_OPTIONS,
)

df_clustering.to_parquet(
    CLUSTERING_PATH,
    **PARQUET_OPTIONS,
)

# ----------------------------------------------------------------
# 12. RELATÓRIO DA EXECUÇÃO
# ----------------------------------------------------------------

print("Processamento concluído.")
print(f"Base original: {df_raw.shape}")
print(f"Base analítica: {df_analytical.shape}")
print(f"Base inicial para agrupamentos: {df_clustering.shape}")
print()
print(f"Auditoria detalhada: {AUDIT_PATH}")
print(f"Resumo da auditoria: {OUTPUT_DIR / 'missing_status_summary.csv'}")
print(f"Base analítica: {ANALYTICAL_PATH}")
print(f"Base para agrupamentos: {CLUSTERING_PATH}")

display(audit_summary.head(65))